# Drift-Sense — Track 1 (SEMICON India Hackathon 2026)
Colab environment for dataset generation, model development, and evaluation.

**Repo:** https://github.com/Shailesh-A-hub/drift-sense-semicon-hackathon-2026

Run cells top to bottom. Re-run the clone cell if teammates have pushed new commits.

## 1. Clone repo + install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/Shailesh-A-hub/drift-sense-semicon-hackathon-2026.git'
REPO_DIR = 'drift-sense-semicon-hackathon-2026'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip install -q -r requirements.txt

## 2. Check GPU availability

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
print('Torch version:', torch.__version__)

## 3. Generate the training dataset (train severity)

In [ ]:
!python dataset_generator/drift_sense_dataset_generator.py \
    --n_pairs 30 --arch both --severity train --out_dir dataset_train --seed0 0

## 4. Generate a held-out, harder eval set (test severity — mimics AMAT's noisier hidden test set)

In [ ]:
!python dataset_generator/drift_sense_dataset_generator.py \
    --n_pairs 30 --arch both --severity test --out_dir dataset_eval --seed0 1000

## 5. Sanity-check + visualize a sample pair

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('dataset_train/manifest.json') as f:
    manifest = json.load(f)

sample = manifest[0]
ref = np.load(sample['reference_path'])
search = np.load(sample['search_path'])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(ref, cmap='gray')
axes[0].set_title(f"Reference ({sample['architecture']})")
axes[1].imshow(search, cmap='gray')
axes[1].scatter([sample['true_x']], [sample['true_y']], c='red', s=80, marker='x')
axes[1].set_title(f"Search — true match at ({sample['true_x']}, {sample['true_y']})")
plt.tight_layout()
plt.savefig('sample_pair_check.png', dpi=100)
plt.show()
print('Total training pairs:', len(manifest))
print('Ambiguity label range:', min(m['ambiguity_label'] for m in manifest), '-', max(m['ambiguity_label'] for m in manifest))

## 6. Next steps (to build here)
- Classical baseline: OpenCV NCC / phase correlation (comparison point for Slide 6)
- Component 1: FFT + autocorrelation lattice normalization
- Components 2-4: LoFTR-style matcher w/ pitch-aware PE, annulus descriptor, confidence head
- `inference.py`: standalone script matching AMAT's exact I/O contract
- Push results back: `!git add -A && git commit -m '...' && git push`